In [1]:
import numpy as np
np.set_printoptions(precision=2, suppress=True)
from homography import Homography
import pandas as pd
from predict import OrientAny

In [2]:
def get_nerf_ccs_to_normal_ccs_T():
    """Get the transformation matrix from NeRF CCS to normal CCS."""
    T = np.asarray([
        [1, 0, 0, 0],
        [0, -1, 0, 0],
        [0, 0, -1, 0],
        [0, 0, 0, 1]
    ])
    return T

In [3]:
CSV = "/robodata/smodak/datasets/orientany/Objaverse_render_random/final_ans/obja_render_train_ex3_dataset.csv"
df = pd.read_csv(CSV)

In [4]:
df.tail()

,uid,angle_ax,angle_pl,has_direction,random_idx
4427595,000-100/fc5c58560f654223afaecb4f3653808c,339.166330,55.323112,True,235
4427596,000-100/fc5c58560f654223afaecb4f3653808c,135.408245,88.562214,True,236
4427597,000-100/fc5c58560f654223afaecb4f3653808c,193.834025,-67.479082,True,237
4427598,000-100/fc5c58560f654223afaecb4f3653808c,261.980643,-34.970161,True,238
4427599,000-100/fc5c58560f654223afaecb4f3653808c,16.090334,-46.073161,True,239


In [5]:
df.head(1)

,uid,angle_ax,angle_pl,has_direction,random_idx
0,000-001/112c059282cf4511a01fd27211edcae8,0.0,0.0,False,160


In [7]:
target_uid = "000-130/f239927bfe2643c1a3e6eb235f58b526"
target_img_idx = 180  # 176, 199, 222, 208, 162
df_filtered = df[(df["uid"] == target_uid) & (df["random_idx"] == target_img_idx)]
print(f"Found {len(df_filtered)} rows")
df_filtered

Found 1 rows


,uid,angle_ax,angle_pl,has_direction,random_idx
165540,000-130/f239927bfe2643c1a3e6eb235f58b526,114.352714,42.927047,True,180


In [8]:
phi = df_filtered["angle_ax"].values[0]
theta_elev = df_filtered["angle_pl"].values[0]

In [9]:
phi, theta_elev

(114.35271350949398, 42.92704692317531)

In [9]:
R_objw_to_normal_ccs_GT = OrientAny.get_R_objw2cam(phi, theta_elev, 0)
R_objw_to_normal_ccs_GT

array([[-0.63,  0.78,  0.  ],
       [ 0.56,  0.45, -0.7 ],
       [-0.54, -0.44, -0.72]])

In [10]:
saved_npy = np.load(f"/robodata/smodak/datasets/orientany/Objaverse_render_random/filter80k_archive_extra3/000-130/filter80k/filter80k_render_extra3/000-130/f239927bfe2643c1a3e6eb235f58b526/random_rt{target_img_idx}.npy")
R_blenderw_to_nerf_ccs = saved_npy[:3, :3]

In [11]:
R_blenderw_to_normal_ccs = get_nerf_ccs_to_normal_ccs_T()[:3, :3] @ R_blenderw_to_nerf_ccs

In [12]:
R_blenderw_to_objw = np.array([
    [-1, 0, 0],
    [0, -1, 0],
    [0, 0, 1]
])

In [13]:
R_objw_to_normal_ccs = R_blenderw_to_normal_ccs @ R_blenderw_to_objw.T
R_objw_to_normal_ccs

array([[-0.63,  0.78, -0.  ],
       [ 0.56,  0.45, -0.7 ],
       [-0.54, -0.44, -0.72]])

In [14]:
# check equality with GT
np.allclose(R_objw_to_normal_ccs, R_objw_to_normal_ccs_GT, atol=1e-6)

True

In [15]:
np.load("/robodata/smodak/datasets/orientany/Objaverse_render_random/filter80k_archive_extra3/000-130/filter80k/filter80k_render_extra3/000-130/f239927bfe2643c1a3e6eb235f58b526/R_blenderw_to_objw176.npy")

array([[-1.,  0., -0.],
       [-0., -1., -0.],
       [-0., -0.,  1.]])